In [1]:
import torch
import torch.nn as nn
from torch.utils.data import DataLoader,TensorDataset
import numpy as np
import os

In [2]:
class TinyMlp(nn.Module):
  def __init__(self):
    super().__init__()
    self.net = nn.Sequential(
        nn.Linear(1,16),
        nn.ReLU(),
        nn.Linear(16,1)
    )
  def forward(self,x):
    return self.net(x)

def getdataloaders():
  torch.manual_seed(42)
  np.random.seed(42)

  X=np.random.rand(1000,1) * 10
  epsilon = np.random.randn(1000,1) * 0.5
  Y= X * 2 + 1 + epsilon

  x_tensor = torch.tensor(X,dtype=torch.float32)
  y_tensor = torch.tensor(Y,dtype=torch.float32)

  train_dataset = TensorDataset(x_tensor[:800],y_tensor[:800])
  val_dataset = TensorDataset(x_tensor[800:],y_tensor[800:])

  train_loader = DataLoader(train_dataset,batch_size=32,shuffle=True)
  val_loader = DataLoader(val_dataset,batch_size=32,shuffle=False)

  return train_loader,val_loader

train_loader,val_loader = getdataloaders()
print(f"训练集批次数: {len(train_loader)}, 验证集批次数: {len(val_loader)}")

训练集批次数: 25, 验证集批次数: 7


In [3]:
def train_one_epoch(model,dataloader,criterion,optimizer):
  model.train()
  total_loss = 0.0

  for x_batch,y_batch in dataloader:
    optimizer.zero_grad()
    preds = model(x_batch)
    loss = criterion(preds,y_batch)
    loss.backward()
    optimizer.step()

    total_loss += loss.item() * x_batch.size(0)
  return total_loss / len(dataloader.dataset)


def evaluate(model,dataloader,criterion):
  model.eval()
  total_loss = 0.0

  with torch.no_grad():
    for x_batch,y_batch in dataloader:
      preds = model(x_batch)
      loss = criterion(preds,y_batch)
      total_loss += loss.item() * x_batch.size(0)

  return total_loss / len(dataloader.dataset)

In [4]:
def save_checkpoint(model,optimizer,epoch,path):
  dir_name = os.path.dirname(path)
  if dir_name and not os.path.exists(dir_name):
    os.mkdir(dir_name)

  checkpoint = {
      'epoch': epoch,
      'model_state_dict': model.state_dict(),
      'optimizer_state_dict': optimizer.state_dict()
  }
  torch.save(checkpoint,path)
  print(f"Checkpoint 已保存至: {path}")

def load_checkpoint(model,optimizer,path):
  if not os.path.exists(path):
    raise FileNotFoundError(f"cant find file")

  checkpoint = torch.load(path,map_location=torch.device('cpu'))

  model.load_state_dict(checkpoint['model_state_dict'])
  optimizer.load_state_dict(checkpoint['optimizer_state_dict'])

  epoch = checkpoint['epoch']
  print(f"Checkpoint 已加载，从 Epoch {epoch} 继续")
  return epoch

In [5]:
model = TinyMlp()
criterion = nn.MSELoss()
optimizer = torch.optim.Adam(model.parameters(),lr=0.01)

epochs = 30

print("开始训练...")
for epoch in range(epochs):
  train_loss = train_one_epoch(model, train_loader, criterion, optimizer)
  val_loss = evaluate(model, val_loader, criterion)

  if (epoch + 1) % 5 == 0 or epoch == 0:
    print(f"Epoch [{epoch+1}/{epochs}] | Train Loss: {train_loss:.4f} | Val Loss: {val_loss:.4f}")
save_checkpoint(model, optimizer, epochs, "tiny_mlp.pth")

开始训练...
Epoch [1/30] | Train Loss: 16.9282 | Val Loss: 3.2307
Epoch [5/30] | Train Loss: 0.2499 | Val Loss: 0.2417
Epoch [10/30] | Train Loss: 0.2466 | Val Loss: 0.2413
Epoch [15/30] | Train Loss: 0.2471 | Val Loss: 0.2420
Epoch [20/30] | Train Loss: 0.2529 | Val Loss: 0.2447
Epoch [25/30] | Train Loss: 0.2529 | Val Loss: 0.2465
Epoch [30/30] | Train Loss: 0.2481 | Val Loss: 0.2601
Checkpoint 已保存至: tiny_mlp.pth


In [12]:
%%writefile test_mlp.py
import pytest
import torch
import torch.nn as nn
import os

class TinyMLp(nn.Module):
  def __init__(self):
    super().__init__()

    self.net = nn.Sequential(
        nn.Linear(1,16),
        nn.ReLU(),
        nn.Linear(16,1)
        )
  def forward(self,x):
    return self.net(x)


def test_output_shape():
  model = TinyMLp()
  x = torch.randn(32,1)
  output = model(x)
  assert output.shape == (32,1)


def test_checkpoint_consistency(tmp_path):

  save_path = os.path.join(tmp_path,"test_ckpt.pth")


  model_a = TinyMLp()
  optimizer = torch.optim.Adam(model_a.parameters())


  torch.save({
      'model_state_dict': model_a.state_dict(),
      'optimizer_state_dict': optimizer.state_dict(),
      'epoch': 1
  },save_path)


  model_b = TinyMLp()
  checkpoint = torch.load(save_path)
  model_b.load_state_dict(checkpoint['model_state_dict'])


  x = torch.randn(10,1)
  out_a = model_a(x)
  out_b = model_b(x)


  assert torch.allclose(out_a,out_b)


def test_eval_does_not_change_weights():
  model = TinyMLp()

  initial_weights = [p.detach().clone() for p in model.parameters()]

  model.eval()
  with torch.no_grad():
    dummy_input = torch.randn(32,1)
    _ = model(dummy_input)


  final_weights = [p.detach().clone() for p in model.parameters()]

  for init_w, final_w in zip(initial_weights, final_weights):
        assert torch.equal(init_w, final_w)

Overwriting test_mlp.py


In [13]:
!pip install pytest -q
!pytest test_mlp.py -v

============================= test session starts ==============================
platform linux -- Python 3.13.18, pytest-8.4.2, pluggy-1.7.0 -- /usr/bin/python3
cachedir: .pytest_cache
rootdir: /content
plugins: langsmith-0.14.17, typeguard-4.6.0, anyio-4.16.0, platformdirs-4.14.3
collected 3 items                                                              

test_mlp.py::test_output_shape PASSED                                    [ 33%]
test_mlp.py::test_checkpoint_consistency PASSED                          [ 66%]
test_mlp.py::test_eval_does_not_change_weights PASSED                    [100%]

============================== 3 passed in 3.52s ===============================
